In [1]:
# ══════════════════════════════════════════════════════════════
# 생활용품 쇼핑몰 리뷰 요약 웹 화면 (Kiwi 형태소 분석 + KNU 한국어 감성사전 + Gradio)
# 대상 파일: TS_5 생활용품 상품평 엑셀 (열: INDEX, 도메인, 카테고리, 상품명, 상품평, 데이터구분)
# ══════════════════════════════════════════════════════════════
#
# [전체 흐름]
#
#  ── 준비 단계 (셀 실행 시 1회) ─────────────────────────────────
#   0. 패키지 설치 : kiwipiepy(형태소 분석), openpyxl(엑셀 읽기), gradio(웹 화면)
#   1. 감성사전 읽기 : KNU 한국어 감성사전 → {단어: 점수(-2 ~ +2)} 딕셔너리 knu
#   2. 감성사전 고치기 : 상품평에 맞지 않는 단어 0점 처리(ignore_words), 단어 추가(add_words)
#   3. 형태소 분석 준비 : Kiwi 생성, 사용자 단어 등록, 토큰 → 단어 목록/명사 목록 변환 함수(to_words)
#   4. 점수 계산 규칙 : 단어 점수(word_score, 부정어 부호 뒤집기), 항목 설정(ASPECTS), 항목 점수(aspect_result)
#   5. 파일 분석 함수 : 엑셀 읽기 → 형태소 분석 → 리뷰 점수·감성 → 항목별 결과 (analyze_file)
#   6. 요약 화면 함수 : 긍정 비율 + 항목별 막대 + 장점·단점 단어 + 대표 리뷰 → HTML (build_page)
#   7. 이벤트 함수 : 파일 선택 시(on_upload), 검색 버튼 클릭 시(on_search)
#   8. 웹 화면 구성·실행 : gr.Blocks로 화면 배치 → demo.launch()
#
#  ── 사용 단계 (웹 화면에서 반복) ───────────────────────────────
#   ① 엑셀 파일 선택  → on_upload  → analyze_file  → 분석 결과를 gr.State에 보관, 카테고리 목록 갱신
#   ② 카테고리·검색어·감성 필터 선택 후 [검색 · 요약] 클릭(또는 검색어 입력 후 Enter)
#                     → on_search  → 조건에 맞는 리뷰만 골라냄
#                     → build_page → 요약 화면 HTML 생성
#   ③ 웹에 결과 표시  → 요약 화면(HTML) + 검색 건수 안내 + 검색된 리뷰 표(Dataframe)
#
#  ── 데이터 흐름 ────────────────────────────────────────────────
#   엑셀(.xlsx) → DataFrame(상품평) → Kiwi 토큰 → 단어 목록 → 감성사전 점수 합계 → 긍정/중립/부정
#                                              └→ 항목 키워드 주변 점수 → 항목별 좋음/보통/아쉬움
#                                              └→ 명사 목록 → 긍정·부정 리뷰의 장점·단점 단어
#
# ══════════════════════════════════════════════════════════════

# 형태소 분석기, 엑셀 읽기 도구, 웹 화면 도구 설치 (처음 한 번만 실행되면 됨)
%pip install -q kiwipiepy openpyxl gradio

# csv 모듈: 감성사전 파일을 읽을 때 따옴표 처리 방식(QUOTE_NONE)을 지정하기 위해 가져옴
import csv
# os 모듈: 감성사전 파일이 현재 폴더에 있는지 확인하기 위해 가져옴
import os
# html 모듈: 리뷰 원문을 HTML에 넣을 때 <, >, & 같은 특수문자를 안전한 글자로 바꾸기 위해 가져옴
import html
# Counter: 단어별 등장 횟수를 세는 딕셔너리 형태의 자료형
from collections import Counter

# pandas: 엑셀 데이터를 표(DataFrame)로 다루는 도구
import pandas as pd
# gradio: 파이썬 함수를 웹 화면(파일 선택, 버튼, 결과 표시)으로 만들어 주는 도구
import gradio as gr
# Kiwi: 한국어 형태소 분석기
from kiwipiepy import Kiwi


# ──────────────────────────────────────────────
# 1. KNU 한국어 감성사전 읽기
# ──────────────────────────────────────────────
# 현재 폴더에 SentiWord_Dict.txt 파일이 있으면 그 파일을, 없으면 GitHub 주소를 읽을 대상으로 정함
if os.path.exists("SentiWord_Dict.txt"):
    source = "SentiWord_Dict.txt"
else:
    source = "https://raw.githubusercontent.com/park1200656/KnuSentiLex/master/SentiWord_Dict.txt"

# 탭(\t)으로 구분된 "단어<탭>점수" 파일을 읽어 두 열짜리 표로 만들고, 빈 값이 있는 행은 제거
#   header=None            : 첫 줄을 열 이름이 아닌 데이터로 읽음
#   names=["단어", "점수"] : 두 열의 이름을 지정
#   quoting=csv.QUOTE_NONE : 따옴표(")를 일반 글자로 취급
lex = pd.read_csv(source, sep="\t", header=None, names=["단어", "점수"], quoting=csv.QUOTE_NONE).dropna()

# 단어 앞뒤 공백을 지우고 점수를 정수로 바꾼 뒤 {단어: 점수} 딕셔너리로 만듦
knu = dict(zip(lex["단어"].str.strip(), lex["점수"].astype(int)))


# ──────────────────────────────────────────────
# 2. 생활용품 상품평에 맞게 감성사전 고치기
# ──────────────────────────────────────────────
# 감성과 관계없이 쓰이거나 상품평에서 뜻이 달라지는 단어 → 0점으로 바꿀 목록
ignore_words = ["바르다", "크다", "선", "이상", "받다", "많다", "가능", "높다", "걸리다", "쓰다", "들다",
                "빠지다", "남다", "날다", "세다", "때"]

# 생활용품(세제, 위생용품, 주방용품) 상품평에 자주 나오는 감성 단어 → {단어: 점수}로 추가 또는 점수 변경
add_words = {"별로": -2, "아쉽다": -1, "불만": -2, "불편": -1, "불편하다": -1,
             "은은하다": 1, "향긋하다": 1, "상쾌하다": 1, "깔끔하다": 1, "깨끗하다": 1,
             "뽀송뽀송하다": 1, "개운하다": 1, "풍성하다": 1, "튼튼하다": 1, "편리하다": 1,
             "간편하다": 1, "가성비": 1, "저렴하다": 1, "싸다": 1, "가볍다": 1, "순하다": 1,
             "퀘퀘하다": -1, "찝찝하다": -1, "독하다": -2, "자극": -1, "트러블": -1,
             "새다": -1, "파손": -2, "냄새나다": -1, "약하다": -1}

# ignore_words의 단어를 하나씩 꺼내 사전에서 0점으로 지정 (없는 단어면 0점으로 새로 추가)
for w in ignore_words:
    knu[w] = 0
# add_words의 {단어: 점수}를 사전에 덮어씀 (있으면 점수 변경, 없으면 추가)
knu.update(add_words)


# ──────────────────────────────────────────────
# 3. Kiwi 형태소 분석기 준비와 단어 목록 만들기 함수
# ──────────────────────────────────────────────
# Kiwi 형태소 분석기 객체 생성
kiwi = Kiwi()

# 잘못 나뉘는 생활용품 단어를 일반명사(NNG) 한 단어로 등록
for word in ["섬유유연제", "주방세제", "세정력", "세척력", "전동칫솔", "물티슈", "가성비", "유통기한"]:
    kiwi.add_user_word(word, "NNG")

# 요약 문장의 장점·단점 단어에서 제외할 흔한 명사 집합
skip = {"제품", "사용", "구매", "구입", "생각", "남편", "신랑", "아이", "아들", "딸", "엄마", "때", "것",
        "만족", "별로", "마음", "처음", "느낌", "정도", "이번", "기존", "선물", "소개", "오늘", "여러분",
        "추천", "도움", "감사", "이웃", "예전", "언어", "하루", "매일", "번", "가지", "부분", "이용", "후기", "리뷰", "다음", "요즘"}

# 형태소 분석 결과(토큰 목록)를 감성사전 모양의 단어 목록과 명사 목록으로 바꾸는 함수
#   반환값 1) words : 감성 점수 계산에 쓰는 단어 목록 (동사·형용사는 '다', 어근은 '하다'를 붙인 모양)
#   반환값 2) nouns : 장점·단점 단어를 찾는 데 쓰는 2글자 이상 일반명사 목록 (skip 단어 제외)
def to_words(tokens):
    words, nouns = [], []
    for t in tokens:
        tag = t.tag
        if tag in {"VV", "VA"} or t.form == "않":
            words.append(t.form + "다")            # 동사·형용사·'않'은 '다'를 붙여 저장
        elif tag == "XR":
            words.append(t.form + "하다")          # 어근은 '하다'를 붙여 저장
        elif tag in {"NNG", "NNP"} or t.form in {"안", "못", "별로"}:
            words.append(t.form)                  # 명사와 부정·평가 표현은 그대로 저장
        if tag == "NNG" and len(t.form) > 1 and t.form not in skip:
            nouns.append(t.form)                  # 2글자 이상 일반명사는 명사 목록에도 저장
    return words, nouns


# ──────────────────────────────────────────────
# 4. 단어 점수와 항목(aspect) 점수 계산 함수
# ──────────────────────────────────────────────
# 단어 "앞"에 와서 뒤 단어의 점수 부호를 뒤집는 부정어 (예: "안 좋다")
FLIP_BEFORE = {"안", "못"}
# 단어 "뒤"에 와서 앞 단어의 점수 부호를 뒤집는 부정어 (예: "불편 없다")
FLIP_AFTER = {"않다", "없다"}

# words 목록에서 j번째 단어의 감성 점수를 계산하는 함수
#   부정어 자체는 0점, 앞에 '안/못' 또는 뒤에 '않다/없다'가 있으면 점수 부호를 뒤집음
def word_score(words, j, lexicon):
    w = words[j]
    if w in FLIP_BEFORE or w in FLIP_AFTER:
        return 0
    value = lexicon.get(w, 0)                                   # 사전에 없는 단어는 0점
    before = words[j - 1] if j > 0 else ""                      # 바로 앞 단어 (없으면 빈 문자열)
    after = words[j + 1] if j + 1 < len(words) else ""          # 바로 뒤 단어 (없으면 빈 문자열)
    if before in FLIP_BEFORE or after in FLIP_AFTER:
        value = -value
    return value

# 생활용품 상품평에 맞춘 평가 항목 설정
#   keywords : 리뷰에서 이 항목을 언급했다고 판단할 단어 집합
#   labels   : 결과 0(좋음), 1(보통), 2(아쉬움)에 대응하는 표시 문구
#   lex      : 이 항목에서만 적용할 단어별 점수 (기본 사전 knu보다 우선 적용)
ASPECTS = {
    "향": {
        "keywords": {"향", "향기", "냄새", "잔향"},
        "labels": ["좋아요", "보통이에요", "아쉬워요"],
        "lex": {"강하다": -1, "진하다": -1, "독하다": -2, "은은하다": 1, "향긋하다": 1, "인공적": -1},
    },
    "품질": {
        "keywords": {"품질", "성능", "효과", "세정력", "세척력", "거품", "내구성", "마감", "흡수력"},
        "labels": ["만족해요", "보통이에요", "아쉬워요"],
        "lex": {"풍성하다": 1, "강력하다": 1, "약하다": -1, "닦이다": 1, "지워지다": 1, "튼튼하다": 1},
    },
    "사용감": {
        "keywords": {"사용감", "편하다", "편리하다", "불편하다", "불편", "간편하다", "그립", "손잡이"},
        "labels": ["편해요", "보통이에요", "불편해요"],
        "lex": {"무겁다": -1, "미끄럽다": -1, "뻑뻑하다": -1},
    },
    "가격": {
        "keywords": {"가격", "가성비", "저렴하다", "비싸다", "싸다"},
        "labels": ["저렴해요", "보통이에요", "비싸요"],
        "lex": {},
    },
    "배송": {
        "keywords": {"배송", "택배", "도착", "포장"},
        "labels": ["빨라요", "보통이에요", "아쉬워요"],
        "lex": {"빠르다": 2, "늦다": -2, "느리다": -2, "걸리다": -1, "총알": 2, "파손": -2,
                "찌그러지다": -2, "새다": -2, "꼼꼼하다": 1},
    },
}

# 항목마다 기본 사전 knu와 항목 전용 점수 lex를 합친 사전을 미리 만들어 둠 (lex가 덮어씀)
ASPECT_LEX = {name: {**knu, **s["lex"]} for name, s in ASPECTS.items()}

# 리뷰 하나(words)에서 한 항목의 평가 결과를 계산하는 함수
#   키워드 위치마다 앞 1개 ~ 뒤 2개 단어의 점수를 합산
#   반환값: 0(합계>0), 1(합계=0), 2(합계<0), None(키워드가 없음)
def aspect_result(words, name):
    setting, lexicon = ASPECTS[name], ASPECT_LEX[name]
    places = [i for i, w in enumerate(words) if w in setting["keywords"]]
    if not places:
        return None
    total = 0
    for i in places:
        for j in range(max(0, i - 1), min(len(words), i + 3)):
            total += word_score(words, j, lexicon)
    return 0 if total > 0 else (2 if total < 0 else 1)


# ──────────────────────────────────────────────
# 5. 엑셀 파일을 읽고 모든 리뷰를 분석하는 함수 (파일 선택 시 1회 실행)
# ──────────────────────────────────────────────
# 엑셀 경로를 받아 리뷰별 단어·명사·점수·항목 결과를 계산한 DataFrame을 반환
def analyze_file(path):
    # 상품평이 비어 있는 행은 제거하고, 상품평을 문자열로 변환
    df = pd.read_excel(path).dropna(subset=["상품평"]).reset_index(drop=True)
    df["상품평"] = df["상품평"].astype(str)
    # 카테고리·상품명 열이 없는 파일이면 빈 문자열 열을 만들어 둠
    for col in ["카테고리", "상품명"]:
        if col not in df.columns:
            df[col] = ""
        df[col] = df[col].fillna("").astype(str)

    # 리뷰 전체를 한 번에 형태소 분석하고, 리뷰마다 (단어 목록, 명사 목록)을 얻음
    pairs = [to_words(tokens) for tokens in kiwi.tokenize(df["상품평"].tolist())]
    df["단어"] = [p[0] for p in pairs]
    df["명사"] = [p[1] for p in pairs]

    # 리뷰마다 모든 단어 점수를 합해 리뷰 점수로 저장
    df["점수"] = [sum(word_score(w, j, knu) for j in range(len(w))) for w in df["단어"]]
    # 리뷰 점수가 0보다 크면 긍정, 0보다 작으면 부정, 0이면 중립으로 표시
    df["감성"] = df["점수"].apply(lambda s: "긍정" if s > 0 else ("부정" if s < 0 else "중립"))

    # 항목마다 리뷰별 평가 결과(0/1/2/None)를 열로 저장
    for name in ASPECTS:
        df[name] = [aspect_result(w, name) for w in df["단어"]]
    return df


# ──────────────────────────────────────────────
# 6. 요약 화면(HTML) 만들기 함수
# ──────────────────────────────────────────────
# 받침 유무에 따라 '은' 또는 '는'을 붙이는 함수 (한글 음절이 아니면 '은/는' 그대로 붙임)
def eun_neun(word):
    last = word[-1]
    if "가" <= last <= "힣":
        return word + ("은" if (ord(last) - 0xAC00) % 28 else "는")
    return word + "은(는)"

# 평가 막대 한 줄(평가 이름, 막대, 리뷰 수, 비율)의 HTML을 만드는 함수
def bar_html(label, count, percent, color):
    return f"""
    <div style="margin:8px 0">
      <div style="font-size:13px">{label}</div>
      <div style="display:flex;align-items:center;gap:8px">
        <div style="flex:1;background:#eee;height:6px;border-radius:3px">
          <div style="width:{percent}%;background:{color};height:6px;border-radius:3px"></div>
        </div>
        <span style="color:#999;font-size:12px">({count:,})</span>
        <b style="width:40px;text-align:right">{percent}%</b>
      </div>
    </div>"""

# 검색된 리뷰 묶음(sub)으로 요약 화면 HTML 전체를 만드는 함수
def build_page(sub, title):
    # 긍정 리뷰 비율(%) : 점수 > 0 인 리뷰의 비율
    positive_rate = round((sub["점수"] > 0).mean() * 100)

    # 항목별로 좋음·보통·아쉬움 개수와 비율을 구해 막대 HTML과 요약 문장 조각을 만듦
    aspect_blocks, parts = [], []
    for name, setting in ASPECTS.items():
        counts = sub[name].value_counts().reindex([0, 1, 2], fill_value=0)   # 0,1,2 순서로 개수 고정
        total = int(counts.sum())                                           # 이 항목을 언급한 리뷰 수
        if total == 0:
            continue                                                        # 언급이 없는 항목은 건너뜀
        percents = [round(counts[k] / total * 100) for k in range(3)]
        bars = "".join(bar_html(setting["labels"][k], int(counts[k]), percents[k],
                                "#f05050" if k == 0 else "#bbb") for k in range(3))
        aspect_blocks.append(f'<div style="flex:1;min-width:150px"><b>{name}</b> '
                             f'<span style="color:#999;font-size:12px">{total:,}건 언급</span>{bars}</div>')
        top = max(range(3), key=lambda k: percents[k])                     # 비율이 가장 높은 평가 번호
        parts.append(f"{eun_neun(name)} '{setting['labels'][top]}' {percents[top]}%")

    # 긍정 리뷰·부정 리뷰의 명사를 세어 상위 10개씩 구하고, 한쪽에만 있는 단어 3개를 장점·단점으로 선정
    pos_top = [w for w, _ in Counter(n for ns in sub[sub["점수"] > 0]["명사"] for n in ns).most_common(10)]
    neg_top = [w for w, _ in Counter(n for ns in sub[sub["점수"] < 0]["명사"] for n in ns).most_common(10)]
    good_words = [w for w in pos_top if w not in neg_top][:3]
    bad_words = [w for w in neg_top if w not in pos_top][:3]

    # 대표 리뷰: 검색된 리뷰 중 점수가 가장 높은 리뷰를 골라 앞 150자만 한 줄로 표시
    best = sub.sort_values("점수", ascending=False)["상품평"].iloc[0].replace("\n", " ").strip()
    best = best[:150] + ("…" if len(best) > 150 else "")

    # 정해진 틀에 계산 결과를 채워 요약 문장 완성 (값이 없는 부분은 문장에서 빠짐)
    summary_text = f"리뷰 {len(sub):,}건 중 {positive_rate}%가 긍정적이에요. "
    if parts:
        summary_text += f"{', '.join(parts)}로 나타났어요. "
    if good_words:
        summary_text += f"좋은 점으로는 {', '.join(good_words)}에 대한 이야기가 많고, "
    if bad_words:
        summary_text += f"아쉬운 점으로는 {', '.join(bad_words)}에 대한 이야기가 많아요. "
    summary_text += f'대표 리뷰: "{html.escape(best)}"'

    # 제목, 긍정 비율, 요약 문장, 항목별 막대를 하나의 HTML로 조립해 반환
    return f"""
    <div style="font-family:sans-serif;max-width:960px;padding:16px;background:#fff;color:#222;border:1px solid #eee;border-radius:8px">
      <div style="text-align:center;font-size:18px;font-weight:bold">리뷰 분석 요약 · {html.escape(title)}</div>
      <div style="text-align:center;color:#f05050;margin:8px 0">긍정리뷰 <span style="font-size:28px;font-weight:bold">{positive_rate}%</span></div>
      <div style="background:#f5f5f5;padding:14px;border-radius:6px;line-height:1.6;font-size:14px">{summary_text}</div>
      <div style="display:flex;gap:20px;flex-wrap:wrap;margin-top:16px">{"".join(aspect_blocks)}</div>
      <div style="color:#999;font-size:11px;margin-top:8px">* Kiwi 형태소 분석 + KNU 한국어 감성사전으로 계산한 결과입니다.</div>
    </div>"""


# ──────────────────────────────────────────────
# 7. Gradio 이벤트 함수
# ──────────────────────────────────────────────
# 검색 결과 표에 보여 줄 열 순서
RESULT_COLS = ["카테고리", "상품명", "감성", "점수", "상품평"]

# [파일 선택 시] 엑셀을 분석해 상태(state)에 저장하고, 카테고리 선택지와 안내 문구를 돌려줌
def on_upload(file):
    if file is None:
        return None, gr.update(choices=["전체"], value="전체"), "엑셀 파일을 선택하세요."
    path = file if isinstance(file, str) else file.name           # Gradio 버전에 따라 경로 문자열 또는 파일 객체
    df = analyze_file(path)
    cats = ["전체"] + sorted(c for c in df["카테고리"].unique() if c)
    msg = f"분석 완료: 리뷰 {len(df):,}건, 카테고리 {len(cats) - 1}개, 상품 {df['상품명'].nunique():,}개"
    return df, gr.update(choices=cats, value="전체"), msg

# [검색 버튼 클릭 시] 카테고리·검색어·감성으로 리뷰를 골라 요약 화면 HTML과 검색 결과 표를 돌려줌
def on_search(df, category, keyword, sentiment):
    empty = pd.DataFrame(columns=RESULT_COLS)
    if df is None:
        return "<p>먼저 엑셀 파일을 선택하세요.</p>", empty, ""

    sub = df
    if category and category != "전체":
        sub = sub[sub["카테고리"] == category]                    # 카테고리 필터
    keyword = (keyword or "").strip()
    if keyword:
        # 검색어가 상품명 또는 상품평에 들어 있는 리뷰만 남김 (정규식 해석 없이 글자 그대로 검색)
        hit = (sub["상품명"].str.contains(keyword, case=False, regex=False)
               | sub["상품평"].str.contains(keyword, case=False, regex=False))
        sub = sub[hit]
    if sub.empty:
        return "<p>검색 결과가 없습니다.</p>", empty, "검색 결과 0건"

    # 요약 화면 제목: 선택한 카테고리와 검색어를 이어 붙임
    title = " / ".join(x for x in [category if category != "전체" else "전체 카테고리", keyword] if x)
    page = build_page(sub, title)

    # 검색 결과 표: 감성 필터를 적용하고 점수 높은 순으로 정렬, 상품평은 줄바꿈을 공백으로 바꿈
    table = sub if sentiment == "전체" else sub[sub["감성"] == sentiment]
    table = table.sort_values("점수", ascending=False)[RESULT_COLS].copy()
    table["상품평"] = table["상품평"].str.replace("\n", " ", regex=False).str.strip()
    info = f"검색된 리뷰 {len(sub):,}건 · 표에 표시 {len(table):,}건 (감성: {sentiment})"
    return page, table, info


# ──────────────────────────────────────────────
# 8. Gradio 웹 화면 구성과 실행
# ──────────────────────────────────────────────
with gr.Blocks(title="생활용품 리뷰 요약") as demo:
    gr.Markdown("## 🧺 생활용품 쇼핑몰 리뷰 요약 (Kiwi + KNU 감성사전)")
    state = gr.State(None)                                           # 분석된 DataFrame을 보관하는 상태 값

    # 전체 흐름 안내 (클릭하면 펼쳐지는 영역)
    with gr.Accordion("📋 전체 흐름 보기", open=False):
        gr.Markdown(
            "| 단계 | 화면에서 할 일 | 내부 처리 | 결과 |\n"
            "|---|---|---|---|\n"
            "| ① 파일 선택 | 엑셀(.xlsx) 선택 | 엑셀 읽기 → Kiwi 형태소 분석 → 감성사전 점수 → 항목별 평가 | 상태 창에 리뷰·카테고리·상품 수 표시 |\n"
            "| ② 조건 선택 | 카테고리, 검색어, 감성 필터 선택 | — | — |\n"
            "| ③ 검색 · 요약 | 버튼 클릭 또는 Enter | 조건에 맞는 리뷰 선택 → 긍정 비율·항목 비율·장단점 단어·대표 리뷰 계산 | 요약 화면 + 검색 결과 표 |\n\n"
            "**점수 규칙**: 단어별 감성사전 점수 합계 > 0 이면 긍정, = 0 이면 중립, < 0 이면 부정 "
            "(앞에 '안·못', 뒤에 '않다·없다'가 오면 부호를 뒤집음)  \n"
            "**항목 평가**: 항목 단어(예: 배송) 앞 1개 ~ 뒤 2개 단어의 점수 합계로 좋음·보통·아쉬움 판정"
        )

    with gr.Row():
        file_in = gr.File(label="① 엑셀 파일 선택 (.xlsx)", file_types=[".xlsx"])
        with gr.Column():
            status = gr.Textbox(label="상태", value="엑셀 파일을 선택하세요.", interactive=False)
            category = gr.Dropdown(label="② 카테고리", choices=["전체"], value="전체")
            keyword = gr.Textbox(label="③ 검색어 (상품명·상품평, 비우면 전체)", placeholder="예: 섬유유연제, 칫솔, 냄새")
            sentiment = gr.Radio(label="④ 결과 표 감성 필터", choices=["전체", "긍정", "중립", "부정"], value="전체")
            btn = gr.Button("🔍 검색 · 요약", variant="primary")

    page_out = gr.HTML()                                             # 요약 화면이 표시되는 영역
    info_out = gr.Markdown()                                         # 검색 건수 안내 문구
    table_out = gr.Dataframe(label="검색 결과", wrap=True)           # 검색된 리뷰 목록 표

    # 파일을 선택하면 on_upload 실행 → 상태·카테고리 선택지·안내 문구 갱신
    file_in.change(on_upload, inputs=file_in, outputs=[state, category, status])
    # 버튼 클릭 또는 검색어 입력 후 Enter → on_search 실행 → 요약 화면·결과 표·안내 문구 갱신
    btn.click(on_search, inputs=[state, category, keyword, sentiment], outputs=[page_out, table_out, info_out])
    keyword.submit(on_search, inputs=[state, category, keyword, sentiment], outputs=[page_out, table_out, info_out])

# 웹 서버 실행 (노트북 셀 아래에 화면이 표시되고, 출력되는 주소로 브라우저에서도 열 수 있음)
demo.launch()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 6.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 32.3 MB/s eta 0:00:00
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a85fb576e038c3598e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
